# Example 6: Error Handling & Retry

演示如何处理 Hy3 API 调用中的常见错误：
- **Timeout**（超时重试）
- **Rate Limit**（429 限流重试）
- **Connection Error**（网络错误重试）
- **指数退避 + Jitter** 通用重试策略

In [ ]:
import time
import random
from openai import OpenAI, APITimeoutError, RateLimitError, APIConnectionError

API_KEY = "sk-你的APIKey"  # TODO: 替换

client = OpenAI(
    base_url="https://tokenhub.tencentmaas.com/v1",
    api_key=API_KEY,
    timeout=30.0,
    max_retries=0,
)

# 指数退避函数
def exponential_backoff(attempt, base_delay=1.0, max_delay=60.0):
    delay = min(base_delay * (2 ** attempt), max_delay)
    jitter = random.uniform(0, delay * 0.1)
    return delay + jitter

# 通用重试装饰器
def retry_with_backoff(max_retries=3, base_delay=1.0, max_delay=60.0):
    retryable = (APITimeoutError, RateLimitError, APIConnectionError)
    def decorator(func):
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except retryable as e:
                    last_exception = e
                    if attempt < max_retries:
                        delay = exponential_backoff(attempt, base_delay, max_delay)
                        print(f"  ⚠️ attempt {attempt+1} failed: {type(e).__name__}")
                        print(f"  ⏳ retry after {delay:.1f}s...")
                        time.sleep(delay)
                    else:
                        print(f"  ❌ max retries reached")
                        raise
            raise last_exception
        return wrapper
    return decorator

## 1. Timeout 处理

使用 1ms timeout 故意触发超时，观察重试逻辑。

In [ ]:
print("Timeout 处理\n")

@retry_with_backoff(max_retries=2, base_delay=1.0)
def call_with_timeout():
    return client.chat.completions.create(
        model="hy3",
        messages=[{"role": "user", "content": "1+1=?"}],
        timeout=0.001,
        extra_body={"reasoning_effort": "no_think"},
    )

try:
    response = call_with_timeout()
    print(f"回答: {response.choices[0].message.content}")
except APITimeoutError:
    print("❌ 最终: 超时，建议增加 timeout 值")

print("\n✅ 正确做法：设置合理 timeout")
response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": "1+1=?"}],
    timeout=60.0,
    extra_body={"reasoning_effort": "no_think"},
)
print(f"回答: {response.choices[0].message.content}")

## 2. 生产级健壮调用

带自动重试和指数退避的 robust chat completion 函数。

In [ ]:
@retry_with_backoff(max_retries=3, base_delay=1.0, max_delay=30.0)
def robust_chat(client, messages, **kwargs):
    return client.chat.completions.create(
        model=kwargs.get("model", "hy3"),
        messages=messages,
        temperature=kwargs.get("temperature", 0.7),
        top_p=kwargs.get("top_p", 1.0),
        timeout=kwargs.get("timeout", 60.0),
        extra_body={"reasoning_effort": kwargs.get("reasoning_effort", "no_think")},
    )

print("生产级调用测试\n")
try:
    response = robust_chat(
        client,
        [{"role": "user", "content": "用一句话说明什么是 API error retry。"}],
        temperature=0.7, timeout=60.0, reasoning_effort="no_think",
    )
    print(f"✅ {response.choices[0].message.content}")
    print(f"📊 prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

## 最佳实践

1. **总是设置 timeout**：根据任务复杂度设 30s~120s
2. **指数退避 + jitter**：避免惊群效应
3. **区分可重试/不可重试错误**：timeout/429/connection 可重试，400/401 不可重试
4. **设置最大重试次数**：建议 3~5 次
5. **直接用 client 内置重试**：`OpenAI(max_retries=3)`